# Anomaly Detection Method Exploration

This notebook explores different ways to determine anomalies in the frozen Forge study panel. A "candidate anomaly" is an unusual observation or change worth inspecting determined by a method, not an established economic explanation, manipulation finding, or formal label.

TODO: remove line below once done with notebook

**Authorship:** the analyst writes the single input SQL statement, substantive feature and analysis code, and graph conclusions. This scaffold provides setup, read-only input access, plotting plumbing, and library I/O orientation. Parameters, formulas, eligibility rules, detector costs, and final methods remain exploratory until agreed during review.

## Study Conventions/Notes

History begins on `2022-06-01`. The existing scoring start of `2022-10-01` is retained for daily scoring methods. PELT is explicitly retrospective and may use full available history.

Due to the way Everef scrapes data from EVE Online's API, ESI, and the fact that we are limiting our scope to the most populated trading hub in EVE Online, we can safely assume that absent source rows mean zero `volume` and zero `order_count`; prices remain null. Specifically, it's due to these components of the [scraper](https://docs.everef.net/commands/scrape-market-history.html):
- `HistoryRegionTypeSource`: any item-region pair traded in the last 450 days will be checked in the order of their market cap, which is the `daily_traded_isk` (`average * volume`) cumulative over history
- `TopTradedRegionTypeSource`: same cumulative `daily_traded_isk`, but also aggregated over regions for each type

Since our selection methods for obtaining our study panel is based on high trading activity (top N `daily_traded_isk` and `order_count`) in Forge, there should be very little risk of our study panel history being missing from this data. Again, we have [the market-history revisions study](everef/market_history_revisions/analysis.ipynb) as a sanity check.

In [1]:
from datetime import date, timedelta

import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
import polars as pl
import pygwalker as pyg
from data_profiling import ProfileReport
from IPython.display import clear_output, display
from plotly.subplots import make_subplots
from skchange.detectors import PELT
from skchange.interval_scorers import L2Cost

from transform.notebook_access import open_analysis_connection

pyg.GlobalVarManager.set_privacy("offline")

HISTORY_START, HISTORY_END = date(2022, 6, 1), date(2026, 9, 1)
SCORING_START = date(2022, 10, 1)
MAX_PLOT_PANELS = 12

# inclusive
def daterange(start_date: date, end_date: date):
    days = int((end_date - start_date).days)
    for n in range(days + 1):
        yield start_date + timedelta(n)


In [2]:
INPUT_SQL= """
    select
        type_id,
        region_id,
        type_name,
        study_category,
        market_date,
        average as vwap,
        highest,
        lowest,
        volume,
        order_count,
        estimated_traded_isk
    from main.int_study_panel_history
    order by type_id, market_date
"""

with open_analysis_connection() as connection:
    panel_history = pl.DataFrame(connection.execute(INPUT_SQL).fetch_arrow_table())

panel_history.head()


In [3]:
panel_history.tail()

In [4]:
panel_history.shape

### Review Plotting

In [5]:
def _require_columns(data: pl.DataFrame, section: str, columns: tuple[str, ...]) -> None:
    missing = [column for column in columns if column not in data.columns]
    if missing:
        raise ValueError(f"{section} requires columns: {', '.join(columns)}. Missing: {', '.join(missing)}.")


def _add_daily_line(figure: go.Figure, rows: pl.DataFrame, value_column: str, name: str, *, row: int | None = None, col: int | None = None, secondary_y: bool | None = None, **kwargs: object) -> None:
    """Add separate Plotly traces so null values and absent calendar days remain gaps."""
    points = rows.select("market_date", value_column).sort("market_date").iter_rows()
    segment_dates, segment_values, segment_number, previous_date = [], [], 0, None
    for market_date, value in points:
        valid = market_date is not None and value is not None and np.isfinite(value)
        contiguous = previous_date is not None and market_date - previous_date == timedelta(days=1)
        if not valid or (segment_dates and not contiguous):
            if segment_dates:
                figure.add_trace(go.Scatter(x=segment_dates, y=segment_values, name=name, legendgroup=name, showlegend=segment_number == 0, mode="lines+markers" if len(segment_dates) == 1 else "lines", **kwargs), row=row, col=col, secondary_y=secondary_y)
                segment_number += 1
            segment_dates, segment_values = [], []
        if valid:
            segment_dates.append(market_date)
            segment_values.append(value)
            previous_date = market_date
        else:
            previous_date = None
    if segment_dates:
        figure.add_trace(go.Scatter(x=segment_dates, y=segment_values, name=name, legendgroup=name, showlegend=segment_number == 0, mode="lines+markers" if len(segment_dates) == 1 else "lines", **kwargs), row=row, col=col, secondary_y=secondary_y)


def show_price_activity(candidate_rows: pl.DataFrame) -> go.Figure:
    _require_columns(candidate_rows, 'Price/activity review', ('type_id', 'type_name', 'market_date', 'vwap', 'volume'))
    groups = candidate_rows.partition_by(['type_id', 'type_name'], maintain_order=True)
    if not groups or len(groups) > MAX_PLOT_PANELS:
        raise ValueError(f'Price/activity review requires between 1 and {MAX_PLOT_PANELS} items.')
    figure = make_subplots(rows=len(groups), cols=1, shared_xaxes=True, specs=[[{"secondary_y": True}] for _ in groups], subplot_titles=[f"{group.get_column('type_name')[0]} ({group.get_column('type_id')[0]})" for group in groups])
    for index, item_data in enumerate(groups, start=1):
        _add_daily_line(figure, item_data, 'vwap', 'VWAP', line={"color": "#1f77b4"}, row=index, col=1, secondary_y=False)
        volume_rows = item_data.filter(pl.col('volume').is_not_null())
        figure.add_trace(go.Bar(x=volume_rows.get_column('market_date'), y=volume_rows.get_column('volume'), name='Volume', legendgroup='Volume', showlegend=index == 1, marker_color='#666666', opacity=0.6), row=index, col=1, secondary_y=True)
        figure.update_yaxes(title_text='VWAP (ISK)', row=index, col=1, secondary_y=False)
        figure.update_yaxes(title_text='Traded units', row=index, col=1, secondary_y=True)
    return figure.update_layout(height=300 * len(groups), barmode='overlay', hovermode='x unified')


def show_method_output(method_rows: pl.DataFrame, output_kind: str) -> go.Figure:
    _require_columns(method_rows, 'Method output', ('type_id', 'type_name', 'market_date', 'method', 'value'))
    if output_kind not in {'daily flag', 'sequential signal', 'retrospective boundary'}:
        raise ValueError('output_kind must identify the method output semantics.')
    groups = method_rows.partition_by(['type_id', 'type_name', 'method'], maintain_order=True)
    if not groups or len(groups) > MAX_PLOT_PANELS:
        raise ValueError(f'Method output requires between 1 and {MAX_PLOT_PANELS} item-method panels.')
    figure = make_subplots(rows=len(groups), cols=1, shared_xaxes=True)
    for index, group in enumerate(groups, start=1):
        figure.add_trace(go.Scatter(x=group.get_column('market_date'), y=group.get_column('value'), mode='markers', name=f"{group.get_column('type_name')[0]} ({group.get_column('type_id')[0]}): {group.get_column('method')[0]}"), row=index, col=1)
    return figure.update_layout(height=280 * len(groups), title=output_kind)


def show_features(rows: pl.DataFrame, feature_columns=("log_return", "log_volume", "relative_range")) -> go.Figure:
    feature_columns = tuple(feature_columns)
    _require_columns(rows, "Feature review", ("type_id", "type_name", "market_date", *feature_columns))
    if not feature_columns:
        raise ValueError("Feature review requires at least one feature column.")
    groups = rows.partition_by(["type_id", "type_name"], maintain_order=True)
    panel_count = len(groups) * len(feature_columns)
    if not groups or panel_count > MAX_PLOT_PANELS:
        raise ValueError(f"Feature review requires between 1 and {MAX_PLOT_PANELS} item-feature panels.")
    figure = make_subplots(rows=panel_count, cols=2, column_titles=['Timeline', 'Distribution'])
    panel_index = 0
    for item_rows in groups:
        for feature in feature_columns:
            row_number = panel_index + 1
            label = f"{item_rows.get_column('type_name')[0]} ({item_rows.get_column('type_id')[0]}): {feature}"
            _add_daily_line(figure, item_rows, feature, label, line={"color": "#1f77b4"}, row=row_number, col=1)
            valid_values = item_rows.get_column(feature).drop_nulls()
            figure.add_trace(go.Histogram(x=valid_values, name=f'{label} distribution', showlegend=False), row=row_number, col=2)
            panel_index += 1
    return figure.update_layout(height=280 * panel_count, barmode='overlay')

## Input, Calendar, and Features

TODO: work on this cell

**Question:** what item-day series and analysis-ready features are needed before detector exploration?

**Inputs:** `panel_history` from the one query, the settled missing-day convention, and the two dates configured above.

**Calculation:** construct applicable item-day calendars; preserve null prices; represent absent applicable activity as zero; and calculate these core features with visible eligibility checks: daily log VWAP return `log(vwap_t / vwap_t-1)` only when adjacent calendar days have valid positive VWAP values; `log(1 + volume)` for nonnegative activity, including settled zero-activity days; and relative high-low range `(highest - lowest) / vwap` when inputs are valid and VWAP is positive. The source-validated high-low range is expected to be sensible. Rolling volatility and multi-day returns are later optional exploration, not required core features.

**Expected output:** an analyst-defined dataframe with feature columns and enough provenance to inspect price and activity together.

In [6]:
# Preserve source rows until the applicable item-day calendar has been constructed.
features = panel_history.with_columns(
    pl.col("market_date").cast(pl.Date, strict=True)
)

features.head()

In [7]:
features.null_count()

In [8]:
features.shape

In [9]:
expected_dates = pl.DataFrame(
    {"market_date": pl.date_range(HISTORY_START, HISTORY_END, "1d", eager=True)}
)
expected_types = features.select("type_id").unique(maintain_order=True)

expected_dates_types = expected_dates.join(expected_types, how="cross")
expected_dates_types.head()

In [10]:
item_mapping = panel_history.select(
    "type_id", "type_name", "study_category"
).unique("type_id", maintain_order=True)
panel_values = features.drop("type_name", "study_category")
features_expected = (
    expected_dates_types.join(
        panel_values, on=["market_date", "type_id"], how="left"
    ).join(item_mapping, on="type_id", how="left")
)
features_expected.null_count()

Looks like we have 625 missing date-type pairs in the data. Due to how Everef scrapes data, and the fact that we are scraping the most popular region in EVE Online, we can safely assume that any missing pairs is a result of no orders/volumes on those date-type pairs. We can't impute the pricing data, so `vwap`, `lowest`, and `highest` will remain NaNs.

In [11]:
features = (
    features_expected.sort("type_id", "market_date")
    .with_columns(
        pl.col("volume").fill_null(0).fill_nan(0),
        pl.col("order_count").fill_null(0).fill_nan(0),
        # Polars uses null for unavailable numeric values; normalize source NaN.
        *[
            pl.when(pl.col(column).is_finite())
            .then(pl.col(column))
            .otherwise(None)
            .alias(column)
            for column in ("vwap", "lowest", "highest")
        ],
    )
    .with_columns(
        (pl.col("vwap") / pl.col("vwap").shift(1).over("type_id"))
        .log()
        .alias("log_return"),
        pl.col("volume").log1p().alias("log_volume"),
        ((pl.col("highest") - pl.col("lowest")) / pl.col("vwap")).alias(
            "relative_range"
        ),
    )
    .with_columns(
        *[
            pl.when(pl.col(column).is_finite())
            .then(pl.col(column))
            .otherwise(None)
            .alias(column)
            for column in ("log_return", "log_volume", "relative_range")
        ]
    )
)

features.head()


In [12]:
features.null_count()

## Rolling MAD

**Worked example:** suppose a prior window of price returns is `[-0.01, 0.00, 0.01, 0.02, 0.00]`. Its median is `0.00`; the median absolute deviation from that median is `0.01`. A new return is assessed by how far it is from the window median relative to that dispersion. A quiet window has a small MAD, so the treatment of zero or near-zero dispersion needs an explicit decision rather than an implicit division.

Use eager Polars item histories and NumPy for the exact MAD calculation. The rolling window, scaling choice, minimum history, feature, threshold, and zero-dispersion behavior are not set here. Natural output: daily flags.

In [13]:
MAD_FEATURES = ("log_return", "log_volume", "relative_range")
COVERAGE_DAYS_REQ = 45
EPSILON = 1e-12
NORMAL_MAD_SCALE = 1.482602218505602

base = features.sort("type_id", "market_date")
if base.select(pl.struct("type_id", "market_date").is_duplicated().any()).item():
    raise ValueError("MAD requires one row per item and calendar date.")

# Long format: one row per item, date, and feature.
id_columns = [column for column in base.columns if column not in MAD_FEATURES]
mad_rows = (
    base.unpivot(
        index=id_columns,
        on=MAD_FEATURES,
        variable_name="feature",
        value_name="feature_value",
    )
    .with_columns(
        pl.when(pl.col("feature_value").is_finite())
        .then(pl.col("feature_value"))
        .otherwise(None)
        .alias("feature_value")
    )
    .sort("type_id", "feature", "market_date")
)
mad_rows.head()


In [14]:
def _rolling_baseline_stats(rows: pl.DataFrame, statistic: str) -> pl.DataFrame:
    """Return prior-60-calendar-day statistics for sorted dense item histories."""
    frames = []
    for group in rows.partition_by(["type_id", "feature"], maintain_order=True):
        values = group.get_column("feature_value").to_numpy().astype(float)
        counts, first, second = [], [], []
        for index in range(len(group)):
            window = values[max(0, index - 60):index]
            window = window[np.isfinite(window)]
            counts.append(len(window))
            if not len(window):
                first.append(None)
                second.append(None)
                continue
            if statistic == "mad":
                center = float(np.median(window))
                first.append(center)
                second.append(float(np.median(np.abs(window - center)) * NORMAL_MAD_SCALE))
            else:
                first.append(float(window.mean()))
                second.append(float(window.std(ddof=1)) if len(window) > 1 else None)
        names = ("baseline_median", "baseline_mad") if statistic == "mad" else ("baseline_mean", "baseline_std")
        frames.append(group.with_columns(
            pl.Series("baseline_count", counts, dtype=pl.Int64),
            pl.Series(names[0], first, dtype=pl.Float64),
            pl.Series(names[1], second, dtype=pl.Float64),
        ))
    return pl.concat(frames, how="vertical")

# [today - 60 days, today): today is excluded. Calculate once for inspection and scoring.
mad_rows = _rolling_baseline_stats(mad_rows, "mad")
baseline_stats = mad_rows.select(
    "type_id", "feature", "market_date", "baseline_count", "baseline_median", "baseline_mad"
)
baseline_stats.head()


In [15]:
enough_history = pl.col("baseline_count") >= COVERAGE_DAYS_REQ
eligible = (
    enough_history
    & pl.col("feature_value").is_not_null()
    & (pl.col("baseline_mad") > EPSILON)
).fill_null(False)
mad_rows = mad_rows.with_columns(
    (enough_history & (pl.col("baseline_mad") <= EPSILON)).fill_null(False).alias("zero_mad"),
    pl.when(eligible)
    .then((pl.col("feature_value") - pl.col("baseline_median")) / pl.col("baseline_mad"))
    .otherwise(None)
    .alias("mad_score"),
)

# Compute baselines with all history, then restrict the scoring period.
rolling_mad_results = mad_rows.filter(pl.col("market_date") >= pl.lit(SCORING_START))
rolling_mad_results.head()


In [16]:
# Use the interactive signal explorer below to select and inspect MAD review rows.
def get_ranked_candidate_table(feature: str, rolling_mad_results: pl.DataFrame) -> pl.DataFrame:
    if feature not in MAD_FEATURES:
        raise ValueError(f"Unknown MAD feature: {feature}")
    return (
        rolling_mad_results
        .filter((pl.col("feature") == feature) & pl.col("mad_score").is_not_null())
        .with_columns(pl.col("mad_score").abs().alias("abs_score"))
        .sort(["abs_score", "market_date"], descending=[True, True])
        .unique("type_id", maintain_order=True)
        .select("type_id", "type_name", "study_category", "market_date", "feature_value", "mad_score", "abs_score", "baseline_count", "baseline_median", "baseline_mad")
    )


In [17]:
REVIEW_FEATURE = "log_return"
candidate_table = get_ranked_candidate_table(REVIEW_FEATURE, rolling_mad_results)
candidate_table.head(10)


### Rolling MAD Review Notes

Write graph conclusions for rolling MAD here: what surfaced, whether price or activity surfaced it, paired-series context, and any observed interpretability or false-positive pattern.

## Rolling Z-score

**Worked example:** if a prior window has mean return `0.01` and standard deviation `0.02`, a new return of `0.05` is `(0.05 - 0.01) / 0.02 = 2` standard deviations above its window mean. The result describes this method's relative surprise, not a score comparable to MAD or PELT output.

Use eager Polars item histories and NumPy mean and standard deviation calculations. The population/sample standard-deviation convention, window, minimum history, feature, and threshold remain unsettled. Natural output: daily flags.

In [18]:
Z_SCORE_FEATURES = MAD_FEATURES
# Z-score uses the fixed sample standard deviation convention (ddof=1).
# Reuse COVERAGE_DAYS_REQ = 45 and EPSILON = 1e-12 from MAD.

z_score_base = features.sort("type_id", "market_date")
if z_score_base.select(pl.struct("type_id", "market_date").is_duplicated().any()).item():
    raise ValueError("Z-score requires one row per item and calendar date.")
z_score_id_columns = [column for column in z_score_base.columns if column not in Z_SCORE_FEATURES]
z_score_rows = (
    z_score_base.unpivot(index=z_score_id_columns, on=Z_SCORE_FEATURES, variable_name="feature", value_name="feature_value")
    .with_columns(pl.when(pl.col("feature_value").is_finite()).then(pl.col("feature_value")).otherwise(None).alias("feature_value"))
    .sort("type_id", "feature", "market_date")
)
z_score_rows.head()


In [19]:
# [today - 60 days, today): today is excluded. Calculate once for inspection and scoring.
z_score_rows = _rolling_baseline_stats(z_score_rows, "z_score")
z_score_baseline_stats = z_score_rows.select(
    "type_id", "feature", "market_date", "baseline_count", "baseline_mean", "baseline_std"
)
z_score_baseline_stats.head()


In [20]:
z_score_enough_history = pl.col("baseline_count") >= COVERAGE_DAYS_REQ
z_score_eligible = (
    z_score_enough_history
    & pl.col("feature_value").is_not_null()
    & (pl.col("baseline_std") > EPSILON)
).fill_null(False)
z_score_rows = z_score_rows.with_columns(
    (z_score_enough_history & (pl.col("baseline_std") <= EPSILON)).fill_null(False).alias("zero_std"),
    pl.when(z_score_eligible)
    .then((pl.col("feature_value") - pl.col("baseline_mean")) / pl.col("baseline_std"))
    .otherwise(None)
    .alias("z_score"),
)
rolling_z_score_results = z_score_rows.filter(pl.col("market_date") >= pl.lit(SCORING_START))
rolling_z_score_results.head()


In [21]:
Z_SCORE_REVIEW_FEATURE = "log_return"
z_score_candidate_table = (
    rolling_z_score_results
    .filter((pl.col("feature") == Z_SCORE_REVIEW_FEATURE) & pl.col("z_score").is_not_null())
    .with_columns(pl.col("z_score").abs().alias("abs_score"))
    .sort(["abs_score", "market_date"], descending=[True, True])
    .select("type_id", "type_name", "study_category", "market_date", "feature_value", "z_score", "abs_score", "baseline_count", "baseline_mean", "baseline_std")
)
z_score_candidate_table.head(10)


### Rolling Z-score Review Notes

Write graph conclusions for rolling Z-score here: what surfaced, whether price or activity surfaced it, paired-series context, and any observed interpretability or false-positive pattern.

## PELT Change Points

**Worked example:** a series with values near `10` for many days and near `20` thereafter may be represented more simply as two stable segments than one. PELT searches for a set of boundaries that balances improved segment fit against the cost of adding boundaries. The returned boundary describes a retrospective change in the observed history, not a day-of-event alarm.

**Agreed first pass:** segment `log(vwap)` and `log_volume` separately over full available history. Compare `L1Cost` (median-level regimes) and `L2Cost` (mean-level regimes); the same runner also accepts `GaussianCost` for later mean/variance exploration. Price runs split at missing or invalid prices and calendar gaps; valid zero-activity days remain in the activity calendar. Missing-value blocks are retained as skipped diagnostics.

Each eligible run is standardized using its full-run mean and sample standard deviation (`ddof=1`). Require at least 28 daily observations, standard deviation above `EPSILON`, and a minimum segment length of 14 observations. Run penalty multipliers `0.5`, `1`, and `2` against explicit base penalties `2 * log(n)` for L1/L2 and `3 * log(n)` for Gaussian, with `penalty_scale=1.0`. This is exploratory sensitivity analysis, not statistically calibrated significance or equally strict penalties across costs. All boundaries and scaling are explicitly retrospective.

Prepare the long-format rows once before the configuration grid. Each daily item-feature run is segmented separately. Boundaries are derived from nonfirst segment starts, so gap edges and run starts are not detected changes. Boundaries and segments retain identifying columns; run configuration, dates, actual penalties, and scaling live in diagnostics. Segment summaries use unstandardized log-feature units. Join tables using `type_id`, `feature`, `cost`, `penalty_multiplier`, and `run_id`. The outer tqdm bar tracks configurations; the inner bar tracks continuous blocks, including skipped blocks. The selected skchange backend is supported by the [retained PELT library benchmark](pelt-library-benchmark/README.md).

#### PELT Helpers

In [22]:
from IPython.display import display
from skchange.detectors import PELT
from skchange.interval_scorers import GaussianCost, L1Cost, L2Cost
from tqdm.auto import tqdm

PELT_ID_COLUMNS = ["type_id", "type_name", "study_category", "feature", "cost", "penalty_multiplier", "run_id"]
PELT_BOUNDARY_COLUMNS = PELT_ID_COLUMNS + ["segment_id", "market_date"]
PELT_SEGMENT_COLUMNS = PELT_ID_COLUMNS + ["segment_id", "segment_start", "segment_end", "n_observations", "feature_mean", "feature_median", "feature_std"]
PELT_DIAGNOSTIC_COLUMNS = PELT_ID_COLUMNS + ["min_segment_length", "min_run_length", "run_start", "run_end", "n_observations", "scaling_mean", "scaling_std", "base_penalty", "penalty", "boundary_count", "status", "skip_reason"]
PELT_PENALTY_PARAMETER_COUNTS = {L1Cost: 1, L2Cost: 1, GaussianCost: 2}


In [23]:
def _prepare_pelt_rows(features: pl.DataFrame) -> pl.DataFrame:
    required = ["type_id", "type_name", "study_category", "market_date", "vwap", "log_volume"]
    base = features.select(required).with_columns(pl.col("market_date").cast(pl.Date, strict=True))
    if base.select(pl.any_horizontal(pl.col("type_id").is_null(), pl.col("market_date").is_null()).any()).item():
        raise ValueError("PELT requires nonmissing item IDs and dates.")
    if base.select(pl.struct("type_id", "market_date").is_duplicated().any()).item():
        raise ValueError("PELT requires one row per item and date.")

    rows = (
        base.with_columns(
            pl.when(pl.col("vwap").is_finite() & (pl.col("vwap") > 0))
            .then(pl.col("vwap").log())
            .otherwise(None)
            .alias("log_vwap")
        )
        .unpivot(
            index=["type_id", "type_name", "study_category", "market_date"],
            on=["log_vwap", "log_volume"],
            variable_name="feature",
            value_name="feature_value",
        )
        .with_columns(
            pl.when(pl.col("feature_value").is_finite())
            .then(pl.col("feature_value"))
            .otherwise(None)
            .alias("feature_value")
        )
        .sort("type_id", "feature", "market_date")
    )
    return rows


def _iter_pelt_runs(rows: pl.DataFrame):
    keys = ["type_id", "feature"]
    valid = pl.col("feature_value").is_not_null()
    starts_run = (
        (pl.col("market_date").diff().over(keys) != pl.duration(days=1))
        | valid.ne(valid.shift(1)).over(keys)
    ).fill_null(True)
    numbered = rows.with_columns(
        starts_run.cast(pl.Int64).cum_sum().over(keys).alias("run_id")
    )
    yield from numbered.partition_by([*keys, "run_id"], maintain_order=True)


def _fit_pelt_run(run: pl.DataFrame, *, cost, penalty_multiplier, min_segment_length, min_run_length):
    values = run.get_column("feature_value").to_numpy().astype(float)
    n = len(values)
    positions = np.array([], dtype=int)
    diagnostics = {"n_observations": n, "scaling_mean": None, "scaling_std": None, "base_penalty": None, "penalty": None, "boundary_count": None, "status": "skipped", "skip_reason": None}
    if not np.isfinite(values).all():
        diagnostics["skip_reason"] = "invalid_feature_values"
        return positions, diagnostics
    diagnostics["scaling_mean"] = float(values.mean())
    diagnostics["scaling_std"] = float(values.std(ddof=1)) if n > 1 else None
    if n < max(min_run_length, 2 * min_segment_length):
        diagnostics["skip_reason"] = "insufficient_history"
        return positions, diagnostics
    if diagnostics["scaling_std"] is None or diagnostics["scaling_std"] <= EPSILON:
        diagnostics["skip_reason"] = "near_zero_dispersion"
        return positions, diagnostics
    standardized = (values - diagnostics["scaling_mean"]) / diagnostics["scaling_std"]
    cost_type = type(cost)
    diagnostics["base_penalty"] = float((PELT_PENALTY_PARAMETER_COUNTS[cost_type] + 1) * np.log(n))
    diagnostics["penalty"] = penalty_multiplier * diagnostics["base_penalty"]
    positions = PELT(cost=cost_type(), penalty=diagnostics["penalty"], penalty_scale=1.0, min_segment_length=min_segment_length, step_size=1).fit(standardized.reshape(-1, 1)).predict(standardized.reshape(-1, 1)).astype(int, copy=False)
    edges = np.concatenate(([0], positions, [n]))
    if np.any(positions <= 0) or np.any(positions >= n) or np.any(np.diff(edges) < min_segment_length):
        raise RuntimeError("PELT returned invalid boundaries or undersized segments.")
    diagnostics.update(boundary_count=len(positions), status="fitted", skip_reason=None)
    return positions, diagnostics


def _summarize_pelt_segments(run: pl.DataFrame, positions: np.ndarray, metadata: dict) -> list[dict]:
    edges = np.concatenate(([0], positions, [run.height]))
    segments = []
    for segment_id, (start, end) in enumerate(zip(edges[:-1], edges[1:]), start=1):
        segment = run.slice(start, end - start)
        values = segment.get_column("feature_value").to_numpy().astype(float)
        segments.append({**metadata, "segment_id": segment_id, "segment_start": segment.get_column("market_date")[0], "segment_end": segment.get_column("market_date")[-1], "n_observations": segment.height, "feature_mean": float(values.mean()), "feature_median": float(np.median(values)), "feature_std": float(values.std(ddof=1)) if len(values) > 1 else None})
    return segments


### PELT Code

In [24]:
def run_pelt(rows: pl.DataFrame, *, cost, penalty_multiplier, min_segment_length=14, min_run_length=28):
    """Segment Polars PELT rows; return boundary, segment, and diagnostic tables."""
    if type(cost) not in PELT_PENALTY_PARAMETER_COUNTS:
        raise ValueError("Use L1Cost, L2Cost, or GaussianCost with estimated parameters.")
    if not np.isfinite(penalty_multiplier) or penalty_multiplier <= 0:
        raise ValueError("penalty_multiplier must be finite and positive.")
    for name, value in [("min_segment_length", min_segment_length), ("min_run_length", min_run_length)]:
        if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < 2:
            raise ValueError(f"{name} must be an integer of at least 2.")

    segment_records, diagnostic_records = [], []
    runs = list(_iter_pelt_runs(rows))
    for run in tqdm(runs, desc=f"{type(cost).__name__}, penalty x{penalty_multiplier:g}", position=1, leave=False):
        positions, diagnostics = _fit_pelt_run(run, cost=cost, penalty_multiplier=penalty_multiplier, min_segment_length=min_segment_length, min_run_length=min_run_length)
        metadata = run.select("type_id", "type_name", "study_category", "feature", "run_id").row(0, named=True)
        metadata.update(cost=type(cost).__name__, penalty_multiplier=penalty_multiplier)
        diagnostic_records.append({**metadata, "min_segment_length": min_segment_length, "min_run_length": min_run_length, "run_start": run.get_column("market_date")[0], "run_end": run.get_column("market_date")[-1], **diagnostics})
        if diagnostics["status"] == "fitted":
            segment_records.extend(_summarize_pelt_segments(run, positions, metadata))

    segments = pl.from_dicts(segment_records, schema=PELT_SEGMENT_COLUMNS)
    boundaries = segments.filter(pl.col("segment_id") > 1).select(
        *PELT_ID_COLUMNS, "segment_id", pl.col("segment_start").alias("market_date")
    )
    return boundaries, segments, pl.from_dicts(diagnostic_records, schema=PELT_DIAGNOSTIC_COLUMNS)


In [25]:
PELT_COSTS = (L1Cost(), L2Cost())
PELT_PENALTY_MULTIPLIERS = (0.5, 1.0, 2.0)
PELT_CONFIGURATIONS = [(cost, multiplier) for cost in PELT_COSTS for multiplier in PELT_PENALTY_MULTIPLIERS]

# Prepare once, then reuse the same input for all six configurations.
pelt_rows = _prepare_pelt_rows(features)
pelt_runs = [run_pelt(pelt_rows, cost=cost, penalty_multiplier=multiplier) for cost, multiplier in tqdm(PELT_CONFIGURATIONS, desc="PELT configurations", position=0)]
pelt_boundaries = pl.concat([result[0] for result in pelt_runs], how="vertical")
pelt_segments = pl.concat([result[1] for result in pelt_runs], how="vertical")
pelt_run_diagnostics = pl.concat([result[2] for result in pelt_runs], how="vertical")
pelt_boundaries.head()


In [ ]:
display(
    pelt_run_diagnostics.group_by("cost", "penalty_multiplier", "feature", "status", "skip_reason", maintain_order=True).len().rename({"len": "run_count"})
)
display(
    pelt_run_diagnostics.filter(pl.col("status") == "fitted").group_by("cost", "penalty_multiplier", "feature", maintain_order=True).agg(
        pl.len().alias("fitted_runs"),
        pl.col("boundary_count").sum().alias("total_boundaries"),
        pl.col("boundary_count").median().alias("median_boundaries_per_run"),
        pl.col("boundary_count").max().alias("maximum_boundaries_per_run"),
    )
)


### PELT Review Notes

Write graph conclusions for PELT here: which retrospective boundaries are useful to inspect, paired price/activity context, and any observed interpretability or segmentation limitation.

## Interactive Signal Explorer

Start with a score method and feature, then restrict the item rows by category and the columns by date. The heatmap color is clipped only for readability: hover retains each exact signed score, baseline diagnostics, actual feature value, and baseline count. Gray cells are unscored rather than zero. Select one item and date for the price/activity and signal drilldown. PELT context is retrospective; its boundaries and segments are not daily scores.

In [ ]:
FEATURE_LABELS = {"log_return": "return", "log_volume": "activity", "relative_range": "range"}
PELT_CONTEXT_FEATURES = {
    "log_return": ("log_vwap", "Price PELT context: log VWAP"),
    "relative_range": ("log_vwap", "Price PELT context: log VWAP; range itself is not segmented"),
    "log_volume": ("log_volume", "Activity PELT context: log volume"),
}


def _score_table(method: str) -> tuple[pl.DataFrame, str, str, str]:
    if method == "MAD":
        return rolling_mad_results, "mad_score", "baseline_median", "baseline_mad"
    return rolling_z_score_results, "z_score", "baseline_mean", "baseline_std"


def _signal_heatmap(score_rows: pl.DataFrame, items: pl.DataFrame, dates: list[date], score_column: str, location_column: str, spread_column: str, clip: float) -> go.Figure:
    item_rows = list(items.select("study_category", "type_id", "type_name").sort(["study_category", "type_id"]).iter_rows(named=True))
    labels = [f"{row['study_category']} | {row['type_id']} | {row['type_name']}" for row in item_rows]
    scores = {
        (row['type_id'], row['market_date']): row
        for row in score_rows.select('type_id', 'market_date', 'feature_value', score_column, 'baseline_count', location_column, spread_column).iter_rows(named=True)
    }
    clipped, unscored, hover = [], [], []
    for row in item_rows:
        clipped_row, unscored_row, hover_row = [], [], []
        for market_date in dates:
            observation = scores.get((row['type_id'], market_date))
            exact_score = observation[score_column] if observation else None
            diagnostics = [None, None, None, None] if observation is None else [observation['feature_value'], observation['baseline_count'], observation[location_column], observation[spread_column]]
            if exact_score is None:
                clipped_row.append(None)
                unscored_row.append(1)
                hover_row.append(['unscored', *diagnostics])
            else:
                clipped_row.append(float(np.clip(exact_score, -clip, clip)))
                unscored_row.append(None)
                hover_row.append([repr(float(exact_score)), *diagnostics])
        clipped.append(clipped_row)
        unscored.append(unscored_row)
        hover.append(hover_row)
    figure = go.Figure()
    hover_template = '%{y}<br>%{x|%Y-%m-%d}<br>Score: %{customdata[0]}<br>Feature value: %{customdata[1]}<br>Baseline count: %{customdata[2]}<br>Baseline location: %{customdata[3]}<br>Baseline spread: %{customdata[4]}<extra></extra>'
    figure.add_trace(go.Heatmap(x=dates, y=labels, z=unscored, customdata=hover, colorscale=[[0, '#d9d9d9'], [1, '#d9d9d9']], showscale=False, hovertemplate=hover_template))
    figure.add_trace(go.Heatmap(x=dates, y=labels, z=clipped, customdata=hover, colorscale='RdBu_r', zmid=0, zmin=-clip, zmax=clip, colorbar={"title": "Clipped score"}, hovertemplate=hover_template))
    return figure.update_layout(height=max(360, 25 * len(labels) + 150), margin={"l": 220, "r": 80, "t": 45, "b": 80}, title='Signed score heatmap; gray = unscored').update_yaxes(autorange='reversed')


def _signal_drilldown(item_features: pl.DataFrame, item_scores: pl.DataFrame, selected_date: date, score_column: str, boundaries: pl.DataFrame) -> go.Figure:
    figure = make_subplots(rows=3, cols=1, shared_xaxes=True, specs=[[{"secondary_y": True}], [{}], [{}]], subplot_titles=['Price and activity', 'Feature value', 'Signed score'])
    _add_daily_line(figure, item_features, 'vwap', 'VWAP', row=1, col=1, secondary_y=False, line={"color": "#1f77b4"})
    volume_rows = item_features.filter(pl.col('volume').is_not_null())
    figure.add_trace(go.Bar(x=volume_rows.get_column('market_date'), y=volume_rows.get_column('volume'), name='Volume', marker_color='#777777', opacity=0.55), row=1, col=1, secondary_y=True)
    _add_daily_line(figure, item_scores, 'feature_value', 'Feature value', row=2, col=1, line={"color": "#2ca02c"})
    _add_daily_line(figure, item_scores, score_column, 'Score', row=3, col=1, line={"color": "#d62728"})
    for boundary_date in boundaries.get_column('market_date').to_list() if boundaries.height else []:
        for row in range(1, 4):
            figure.add_vline(x=boundary_date, line_dash='dot', line_color='#9467bd', opacity=0.7, row=row, col=1)
    for row in range(1, 4):
        figure.add_vline(x=selected_date, line_dash='dash', line_color='#111111', row=row, col=1)
    return figure.update_layout(height=780, barmode='overlay', hovermode='x unified')


def _pelt_context_rows(rows: pl.DataFrame, start: date, end: date, start_column: str, end_column: str) -> pl.DataFrame:
    """Return overlapping PELT rows plus the immediately adjacent rows for context."""
    overlapping = rows.filter((pl.col(end_column) >= start) & (pl.col(start_column) <= end))
    before = rows.filter(pl.col(end_column) < start).sort(end_column).tail(1)
    after = rows.filter(pl.col(start_column) > end).sort(start_column).head(1)
    return pl.concat([before, overlapping, after], how='vertical').unique(maintain_order=True)


def build_signal_explorer() -> widgets.VBox:
    categories = sorted(features.get_column('study_category').unique().to_list())
    all_items = features.select('study_category', 'type_id', 'type_name').unique().sort(['study_category', 'type_id'])
    date_min, date_max = features.select(pl.col('market_date').min().alias('date_min'), pl.col('market_date').max().alias('date_max')).row(0)
    method = widgets.ToggleButtons(options=['MAD', 'Z'], description='Method:')
    feature = widgets.Dropdown(options=[(label, name) for name, label in FEATURE_LABELS.items()], description='Feature:')
    categories_widget = widgets.SelectMultiple(options=categories, description='Categories:')
    start = widgets.DatePicker(value=SCORING_START, description='Start:')
    end = widgets.DatePicker(value=date_max, description='End:')
    item = widgets.Dropdown(description='Item:')
    selected_date = widgets.DatePicker(value=date_max, description='Inspect date:')
    clip = widgets.FloatSlider(value=5.0, min=1.0, max=15.0, step=0.5, description='Color clip:')
    costs = ['None', *sorted(pelt_run_diagnostics.get_column('cost').unique().to_list())]
    pelt_cost = widgets.Dropdown(options=costs, description='PELT cost:')
    pelt_multiplier = widgets.Dropdown(options=sorted(pelt_run_diagnostics.get_column('penalty_multiplier').unique().to_list()), description='Multiplier:')
    output = widgets.Output()

    def render(_change=None) -> None:
        with output:
            clear_output(wait=True)
            if start.value is None or end.value is None or start.value > end.value:
                display(widgets.HTML('<b>Choose a valid start and end date.</b>'))
                return
            if item.value is None or selected_date.value is None:
                display(widgets.HTML('<b>Choose an item and inspection date.</b>'))
                return
        scores, score_column, location_column, spread_column = _score_table(method.value)
        category_expression = pl.lit(True) if not categories_widget.value else pl.col('study_category').is_in(categories_widget.value)
        score_selection = scores.filter(category_expression & (pl.col('feature') == feature.value) & pl.col('market_date').is_between(start.value, end.value))
        selected_items = all_items.filter(category_expression)
        if score_selection.is_empty() or selected_items.is_empty():
            with output:
                display(widgets.HTML('<b>No scored rows match the selected filters.</b>'))
            return
        date_columns = pl.date_range(start.value, end.value, '1d', eager=True).to_list()
        item_score_rows = score_selection.filter(pl.col('type_id') == item.value)
        item_feature_rows = features.filter((pl.col('type_id') == item.value) & pl.col('market_date').is_between(start.value, end.value))
        pelt_feature, pelt_label = PELT_CONTEXT_FEATURES[feature.value]
        base_boundaries = pelt_boundaries.filter((pl.col('type_id') == item.value) & (pl.col('feature') == pelt_feature) & (pl.col('cost') == pelt_cost.value) & (pl.col('penalty_multiplier') == pelt_multiplier.value)) if pelt_cost.value != 'None' else pelt_boundaries.head(0)
        boundary_selection = _pelt_context_rows(base_boundaries, start.value, end.value, 'market_date', 'market_date') if pelt_cost.value != 'None' else base_boundaries
        diagnostics = item_score_rows.filter(pl.col('market_date') == selected_date.value).select('market_date', 'feature_value', score_column, 'baseline_count', location_column, spread_column)
        base_segments = pelt_segments.filter((pl.col('type_id') == item.value) & (pl.col('feature') == pelt_feature) & (pl.col('cost') == pelt_cost.value) & (pl.col('penalty_multiplier') == pelt_multiplier.value)) if pelt_cost.value != 'None' else pelt_segments.head(0)
        segments = _pelt_context_rows(base_segments, start.value, end.value, 'segment_start', 'segment_end') if pelt_cost.value != 'None' else base_segments
        with output:
            display(_signal_heatmap(score_selection, selected_items, date_columns, score_column, location_column, spread_column, clip.value))
            display(diagnostics)
            display(_signal_drilldown(item_feature_rows, item_score_rows, selected_date.value, score_column, boundary_selection))
            if pelt_cost.value != 'None':
                display(widgets.HTML(f'<b>{pelt_label}</b>'))
                display(segments)

    def update_item_options(_change=None) -> None:
        scoped_items = all_items if not categories_widget.value else all_items.filter(pl.col('study_category').is_in(categories_widget.value))
        options = [(f"{row['study_category']} | {row['type_id']} | {row['type_name']}", row['type_id']) for row in scoped_items.iter_rows(named=True)]
        item.options = options
        if options and item.value not in {option[1] for option in options}:
            item.value = options[0][1]

    categories_widget.observe(update_item_options, names='value')
    update_item_options()
    for control in (method, feature, categories_widget, start, end, item, selected_date, clip, pelt_cost, pelt_multiplier):
        control.observe(render, names='value')
    render()
    return widgets.VBox([widgets.HBox([method, feature, clip]), widgets.HBox([categories_widget, start, end]), widgets.HBox([item, selected_date]), widgets.HBox([pelt_cost, pelt_multiplier]), output])



In [ ]:
signal_explorer = build_signal_explorer()
display(signal_explorer)

## Full-data Explorers

PyGWalker launches only after its button is pressed and receives a pandas copy only at that library boundary. No sampling is applied. Seven daily explorer views are provided: one default Market + features + MAD/Z view, plus the same daily base joined to PELT context for each of the six PELT cost-by-penalty configurations. PELT boundaries, segments, and diagnostics also remain available as separate raw dataframes. The profile is on demand, limited to one selected item and selected columns. All item and date filtering happens inside PyGWalker.

In [ ]:
from IPython.display import clear_output, display

_daily_score_keys = ["type_id", "market_date", "feature"]
if rolling_mad_results.select(pl.struct(_daily_score_keys).is_duplicated().any()).item() or rolling_z_score_results.select(pl.struct(_daily_score_keys).is_duplicated().any()).item():
    raise ValueError("MAD/Z comparison requires one row per item, day, and feature.")
mad_comparison = rolling_mad_results.select(*_daily_score_keys, "type_name", "study_category", "feature_value", "baseline_count", "baseline_median", "baseline_mad", "zero_mad", "mad_score").rename({"feature_value": "mad_feature_value", "baseline_count": "mad_baseline_count"})
z_comparison = rolling_z_score_results.select(*_daily_score_keys, "feature_value", "baseline_count", "baseline_mean", "baseline_std", "zero_std", "z_score").rename({"feature_value": "z_feature_value", "baseline_count": "z_baseline_count"})
mad_z_comparison = mad_comparison.join(z_comparison, on=_daily_score_keys, how="inner", validate="1:1")


def _baseline_long_to_wide(long_rows: pl.DataFrame, prefix: str, stat_columns: list[str]) -> pl.DataFrame:
    keys = ["type_id", "market_date", "feature"]
    if long_rows.select(pl.struct(keys).is_duplicated().any()).item():
        raise ValueError("MAD/Z explorer baselines require unique item, day, and feature keys.")
    selected = long_rows.select(*keys, *stat_columns)
    wide = None
    for feature_name in sorted(selected.get_column("feature").unique().to_list()):
        part = selected.filter(pl.col("feature") == feature_name).select(
            "type_id",
            "market_date",
            *[pl.col(column).alias(f"{prefix}_{column}_{feature_name}") for column in stat_columns if column in selected.columns],
        )
        wide = part if wide is None else wide.join(part, on=["type_id", "market_date"], how="full", coalesce=True)
    scoring_start = SCORING_START
    for column in list(wide.columns):
        if "score" in column:
            wide = wide.with_columns(
                pl.when(pl.col("market_date") >= pl.lit(scoring_start))
                .then(pl.col(column))
                .otherwise(None)
                .alias(column)
            )
    return wide


def _default_daily_explorer_view() -> pl.DataFrame:
    work = features.clone()
    base_keys = ["type_id", "market_date"]
    if work.select(pl.struct(base_keys).is_duplicated().any()).item():
        raise ValueError("Explorer requires unique item-day market rows.")
    selected_columns = [
        "market_date", "region_id", "type_id", "type_name", "study_category",
        "vwap", "daily_high", "daily_low", "volume", "order_count",
        "estimated_traded_isk", "log_return", "log_volume", "relative_range",
    ]
    daily = work.with_columns(
        pl.col("highest").alias("daily_high"),
        pl.col("lowest").alias("daily_low"),
    ).select(selected_columns)
    mad_wide = _baseline_long_to_wide(mad_rows, "mad", ["baseline_count", "baseline_median", "baseline_mad", "zero_mad", "mad_score"])
    z_wide = _baseline_long_to_wide(z_score_rows, "z", ["baseline_count", "baseline_mean", "baseline_std", "zero_std", "z_score"])
    daily = daily.join(mad_wide, on=base_keys, how="left", validate="1:1")
    daily = daily.join(z_wide, on=base_keys, how="left", validate="1:1")
    return daily


def _expand_interval(frame: pl.DataFrame, start_column: str, end_column: str) -> pl.DataFrame:
    return frame.filter(pl.col(start_column).is_not_null() & pl.col(end_column).is_not_null() & (pl.col(start_column) <= pl.col(end_column))).with_columns(
        pl.date_ranges(start_column, end_column, interval="1d").alias("market_date")
    ).explode("market_date")


def _pelt_config_ids() -> list[tuple[str, float]]:
    configurations = [(type(cost).__name__, float(multiplier)) for cost, multiplier in PELT_CONFIGURATIONS]
    if len(configurations) != 6 or len(set(configurations)) != 6:
        raise ValueError("Explorer requires the six configured PELT cost/penalty combinations.")
    return configurations


def _attach_pelt_context(base: pl.DataFrame, cost_name: str, multiplier: float) -> pl.DataFrame:
    result = base.clone()
    diagnostics = pelt_run_diagnostics
    boundaries = pelt_boundaries
    segments = pelt_segments
    for feature_name in ("log_vwap", "log_volume"):
        prefix = f"{feature_name}_pelt_"
        result = result.with_columns(
            pl.lit(None).cast(pl.Int64).alias(prefix + "run_id"),
            pl.lit(None).cast(pl.Int64).alias(prefix + "min_segment_length"),
            pl.lit(None).cast(pl.Int64).alias(prefix + "min_run_length"),
            pl.lit(None).cast(pl.Date).alias(prefix + "run_start"),
            pl.lit(None).cast(pl.Date).alias(prefix + "run_end"),
            pl.lit(None).cast(pl.Int64).alias(prefix + "run_n_observations"),
            pl.lit(None).cast(pl.Float64).alias(prefix + "scaling_mean"),
            pl.lit(None).cast(pl.Float64).alias(prefix + "scaling_std"),
            pl.lit(None).cast(pl.Float64).alias(prefix + "base_penalty"),
            pl.lit(None).cast(pl.Float64).alias(prefix + "penalty"),
            pl.lit(None).cast(pl.Int64).alias(prefix + "diagnostic_boundary_count"),
            pl.lit(None).cast(pl.String).alias(prefix + "run_status"),
            pl.lit(None).cast(pl.String).alias(prefix + "skip_reason"),
            pl.lit(None).cast(pl.Int64).alias(prefix + "segment_id"),
            pl.lit(None).cast(pl.Date).alias(prefix + "segment_start"),
            pl.lit(None).cast(pl.Date).alias(prefix + "segment_end"),
            pl.lit(None).cast(pl.Int64).alias(prefix + "segment_n_observations"),
            pl.lit(None).cast(pl.Float64).alias(prefix + "segment_mean"),
            pl.lit(None).cast(pl.Float64).alias(prefix + "segment_median"),
            pl.lit(None).cast(pl.Float64).alias(prefix + "segment_std"),
            pl.lit(None).cast(pl.Boolean).alias(prefix + "boundary"),
        )
    diag = diagnostics.filter((pl.col("cost") == cost_name) & (pl.col("penalty_multiplier") == float(multiplier)))
    boundary_frame = None
    if not boundaries.is_empty():
        boundary_frame = boundaries.filter((pl.col("cost") == cost_name) & (pl.col("penalty_multiplier") == float(multiplier)))
    segment_frame = None
    if not segments.is_empty():
        segment_frame = segments.filter((pl.col("cost") == cost_name) & (pl.col("penalty_multiplier") == float(multiplier)))
    for feature_name in ("log_vwap", "log_volume"):
        prefix = f"{feature_name}_pelt_"
        feature_diag = diag.filter(pl.col("feature") == feature_name)
        run_daily = None
        if not feature_diag.is_empty():
            run_daily = _expand_interval(feature_diag, "run_start", "run_end")
            rename_map = {
                "run_id": prefix + "run_id",
                "min_segment_length": prefix + "min_segment_length",
                "min_run_length": prefix + "min_run_length",
                "run_start": prefix + "run_start",
                "run_end": prefix + "run_end",
                "n_observations": prefix + "run_n_observations",
                "scaling_mean": prefix + "scaling_mean",
                "scaling_std": prefix + "scaling_std",
                "base_penalty": prefix + "base_penalty",
                "penalty": prefix + "penalty",
                "boundary_count": prefix + "diagnostic_boundary_count",
                "status": prefix + "run_status",
                "skip_reason": prefix + "skip_reason",
            }
            run_daily = run_daily.select([column for column in run_daily.columns if column in ("type_id", "market_date", *rename_map.keys())])
            run_daily = run_daily.rename({old: new for old, new in rename_map.items() if old in run_daily.columns})
            if run_daily.select(pl.struct("type_id", "market_date").is_duplicated().any()).item():
                raise ValueError("PELT diagnostic intervals overlap for an item-day.")
        feature_segments = segment_frame
        if feature_segments is not None and not feature_segments.is_empty():
            feature_segments = feature_segments.filter(pl.col("feature") == feature_name)
        segment_daily = None
        if feature_segments is not None and not feature_segments.is_empty():
            segment_daily = _expand_interval(feature_segments, "segment_start", "segment_end")
            segment_daily = segment_daily.select(
                "type_id",
                "market_date",
                pl.col("segment_id").alias(prefix + "segment_id"),
                pl.col("segment_start").alias(prefix + "segment_start"),
                pl.col("segment_end").alias(prefix + "segment_end"),
                pl.col("n_observations").alias(prefix + "segment_n_observations"),
                pl.col("feature_mean").alias(prefix + "segment_mean"),
                pl.col("feature_median").alias(prefix + "segment_median"),
                pl.col("feature_std").alias(prefix + "segment_std"),
            )
            if segment_daily.select(pl.struct("type_id", "market_date").is_duplicated().any()).item():
                raise ValueError("PELT segment intervals overlap for an item-day.")
        boundary_daily = None
        if boundary_frame is not None and not boundary_frame.is_empty():
            selected_boundaries = boundary_frame.filter(pl.col("feature") == feature_name)
            if not selected_boundaries.is_empty():
                boundary_daily = selected_boundaries.select("type_id", "market_date")
                if boundary_daily.select(pl.struct("type_id", "market_date").is_duplicated().any()).item():
                    raise ValueError("PELT boundary rows must be unique per feature, configuration, item, and day.")
                boundary_daily = boundary_daily.with_columns(pl.lit(True).alias(prefix + "boundary_source"))
        if run_daily is not None:
            result = result.drop([column for column in run_daily.columns if column not in ("type_id", "market_date")]).join(run_daily, on=["type_id", "market_date"], how="left", validate="1:1")
        if segment_daily is not None:
            result = result.drop([column for column in segment_daily.columns if column not in ("type_id", "market_date")]).join(segment_daily, on=["type_id", "market_date"], how="left", validate="1:1")
        if boundary_daily is not None:
            result = result.join(boundary_daily, on=["type_id", "market_date"], how="left", validate="1:1")
        source_column = prefix + "boundary_source"
        status_column = prefix + "run_status"
        if status_column in result.columns:
            if source_column in result.columns:
                result = result.with_columns(
                    pl.when(pl.col(status_column) == "fitted")
                    .then(pl.col(source_column).fill_null(False))
                    .otherwise(None)
                    .alias(prefix + "boundary")
                ).drop(source_column)
            else:
                result = result.with_columns(pl.when(pl.col(status_column) == "fitted").then(pl.lit(False)).otherwise(None).alias(prefix + "boundary"))
    return result


_pelt_explorer_configs = _pelt_config_ids()
_pelt_explorer_names = [
    f"Market + features + MAD/Z + PELT context: {cost_name}, penalty x{multiplier:g}"
    for cost_name, multiplier in _pelt_explorer_configs
]
all_computed02_datasets: dict[str, object] = {"Market + features + MAD/Z": _default_daily_explorer_view}
for _view_name, (_cost_name, _multiplier) in zip(_pelt_explorer_names, _pelt_explorer_configs, strict=True):
    all_computed02_datasets[_view_name] = lambda cost_name=_cost_name, multiplier=_multiplier: _attach_pelt_context(_default_daily_explorer_view(), cost_name, multiplier)

walker_dataset = widgets.Dropdown(options=list(all_computed02_datasets), description="Dataset:")
walker_button = widgets.Button(description="Launch PyGWalker", button_style="primary")
walker_output = widgets.Output()


def launch_computed02_walker(_button) -> None:
    with walker_output:
        clear_output(wait=True)
        data = all_computed02_datasets[walker_dataset.value]
        if callable(data):
            data = data()
        if not isinstance(data, pl.DataFrame):
            display(widgets.HTML("<b>The selected explorer view is unavailable.</b>"))
            return
        if data.is_empty():
            display(widgets.HTML("<b>No rows match the selected PyGWalker view.</b>"))
            return
        display(pyg.walk(data.to_pandas(), env="JupyterWidget", cloud_computation=False, show_cloud_tool=False))


walker_button.on_click(launch_computed02_walker)

profile_item = widgets.Dropdown(options=[(f"{row['study_category']} | {row['type_id']} | {row['type_name']}", row['type_id']) for row in features.select('study_category', 'type_id', 'type_name').unique().sort(['study_category', 'type_id']).iter_rows(named=True)], description='Item:')
profile_start = widgets.DatePicker(value=None, description='Start:')
profile_end = widgets.DatePicker(value=None, description='End:')
profile_features = widgets.SelectMultiple(options=list(FEATURE_LABELS), value=('log_return', 'log_volume', 'relative_range'), description='Features:')
profile_button = widgets.Button(description='Build ProfileReport', button_style='primary')
profile_output = widgets.Output()


def build_selected_profile(_button) -> None:
    with profile_output:
        clear_output(wait=True)
        if profile_item.value is None:
            display(widgets.HTML('<b>Choose one item to profile.</b>'))
            return
        if profile_start.value and profile_end.value and profile_start.value > profile_end.value:
            display(widgets.HTML('<b>Start date must not be after end date.</b>'))
            return
        columns = ['market_date', 'type_id', 'type_name', 'study_category', 'vwap', 'volume', 'order_count', *profile_features.value]
        data = features.filter(pl.col('type_id') == profile_item.value)
        if profile_start.value:
            data = data.filter(pl.col('market_date') >= profile_start.value)
        if profile_end.value:
            data = data.filter(pl.col('market_date') <= profile_end.value)
        data = data.select(columns)
        if data.is_empty():
            display(widgets.HTML('<b>No rows match the selected profile filters.</b>'))
            return
        # The selected one-item time series is the sole ProfileReport pandas boundary.
        ProfileReport(data.to_pandas(), minimal=True, title='Selected item time-series profile').to_notebook_iframe()


profile_button.on_click(build_selected_profile)
display(widgets.VBox([widgets.HTML('<b>PyGWalker</b>'), walker_dataset, walker_button, walker_output, widgets.HTML('<b>Selected time-series ProfileReport</b>'), profile_item, profile_start, profile_end, profile_features, profile_button, profile_output]))

## Candidate Review and Case Studies

**Question:** which method-identified observations or boundaries warrant inspection, and what does the paired price/activity context show?

**Inputs:** one method's results at a time, `features`, and the original market fields.

**Calculation:** select candidates within a method, retain whether price or activity surfaced each candidate, and annotate the other series. Keep daily flags, sequential signal dates, and retrospective boundaries in separate review tables.

**Expected output:** a small synthesis dataframe and focused case-study plots drawing on the method-specific reviews above. Record review notes; do not assign formal labels or compare raw scores across methods.

In [ ]:
# TODO (analyst): synthesize selected candidates from completed method reviews into case-study rows.
# State whether price or activity surfaced each candidate and annotate the other measure.
candidate_review: pl.DataFrame | None = None

## Review Notes and Conclusions

Write graph conclusions here after reviewing candidates. Address interpretability, false-positive patterns observed in context, and the distinction between short shocks and persistent changes. Select one or two defensible economic case studies only if the evidence supports them; do not infer causation from detector output alone.

## Optional: 7/14/30 Calendar-day Follow-up

For reviewed candidates, optionally describe price and activity at 7, 14, and 30 calendar days after the natural output date. These are descriptive follow-up horizons, not detector windows, event durations, or mandatory persistence labels. The analyst decides the calculation and treatment of unavailable prices.

In [ ]:
# TODO (analyst, optional): calculate descriptive 7/14/30 calendar-day follow-up context for reviewed candidates.
follow_up_context: pl.DataFrame | None = None

## Optional and Deferred: CUSUM

CUSUM remains optional and deferred. A fixed-reference sequential CUSUM requires explicit agreement on its reference value, allowance, threshold, and reset behavior; no verified library is being silently substituted. If introduced later, it should emit sequential signal dates, distinct from daily flags and retrospective PELT boundaries.

In [ ]:
# TODO (analyst, deferred): document agreed CUSUM reference, allowance, threshold, and reset before implementation.
# No CUSUM recurrence is implemented in this scaffold.
cusum_signals: pl.DataFrame | None = None

## Method Decisions and Later SQL Parity

After review, record candidate-review and case-study decisions: retained feature(s), method-specific parameters, eligibility rules, output semantics, and limitations. This notebook does not choose them in advance.

Any later SQL implementation must preserve the selected method's calendar construction, missing-price behavior, feature eligibility, exact rolling or sequential semantics, thresholding, and output type. Validate Python/SQL parity on a bounded sample before publication; do not replace the analytical pipeline with notebook batch output.